# Day 3.6 — Tools with Side Effects
Day 1's tools only computed answers. Today's change things — and reading, drafting, sending and
deleting are four different risks, classified by the engineer *before* any tool is exposed.

### Step 1 — A simulated workspace and four classified tools

No real calendar, no real email account: every side effect is a list you can print. Registering
with Day 1's `make_tool` gives the model a schema and validates arguments before anything runs.

In [ ]:
class SimulatedWorkspace:
    """Four lists standing in for a real account, so every side effect is visible."""

    def __init__(self):
        self.calendar = [{"time": "10:00", "title": "Project review"},
                         {"time": "15:00", "title": "Lab practice"}]
        self.tasks = ["Finish lab notes", "Review citations"]
        self.drafts, self.sent = [], []

    def view_calendar(self):
        return self.calendar                                   # a read: returns, changes nothing

    def create_draft(self, to, subject, body):
        self.drafts.append({"to": to, "subject": subject, "body": body})
        return f"draft saved for {to}"                          # local, reversible

    def send_email(self, to, subject, body):
        self.sent.append({"to": to, "subject": subject, "body": body})
        return f"message delivered to {to}"                     # leaves the machine

    def delete_all_tasks(self):
        count = len(self.tasks)
        self.tasks.clear()
        return f"deleted {count} tasks"                         # destructive

class NoArgs(BaseModel):
    pass

class MessageArgs(BaseModel):
    to: str = Field(min_length=3, max_length=80)
    subject: str = Field(min_length=1, max_length=120)
    body: str = Field(min_length=1, max_length=2000)

# The risk class is an engineering decision, written down once, before anything is exposed.
TOOL_RISK = {"view_calendar": "read-only", "create_draft": "reversible local write",
             "send_email": "external action", "delete_all_tasks": "destructive"}

def workspace_tools(workspace):
    """Register the four tools in Day 1's format: schema + argument model + function."""
    return {
        "view_calendar": make_tool("view_calendar", "Show the user's calendar for today.",
                                   workspace.view_calendar, NoArgs),
        "create_draft": make_tool("create_draft", "Save an email draft locally. Does not send.",
                                  workspace.create_draft, MessageArgs),
        "send_email": make_tool("send_email", "Send an email. This cannot be undone.",
                                workspace.send_email, MessageArgs),
        "delete_all_tasks": make_tool("delete_all_tasks", "Delete every task in the user's list.",
                                      workspace.delete_all_tasks, NoArgs),
    }

def show(workspace, label):
    print(f"{label:<28} calendar={len(workspace.calendar)} tasks={len(workspace.tasks)} "
          f"drafts={len(workspace.drafts)} sent={len(workspace.sent)}")

workspace = SimulatedWorkspace()
TOOLS = workspace_tools(workspace)
for name, risk in TOOL_RISK.items():
    print(f"{name:<18} {risk}")
print()
show(workspace, "starting state:")

### Step 2 — Four direct calls, four different consequences

We call the functions ourselves. Reading twice leaves the state identical; a draft can be thrown
away; a send cannot be recalled; the delete takes the tasks with it.

In [ ]:
show(workspace, "before anything:")
workspace.view_calendar(); workspace.view_calendar()          # a read, twice
show(workspace, "after two reads:")

workspace.create_draft("mentor@example.test", "Update", "Synthetic only")
show(workspace, "after create_draft:")

workspace.send_email("mentor@example.test", "Update", "Synthetic only")
show(workspace, "after send_email:")

print("\ntasks before delete:", workspace.tasks)
print("returned            :", workspace.delete_all_tasks())
print("tasks after delete  :", workspace.tasks)
print("\nNo check ran, and nothing was recorded. A tool description saying 'dangerous, ask first'")
print("would have changed nothing: descriptions guide a model's choice, they are not a boundary.")

### Step 3 — Day 1's agent loop is not enough

Give the loop these tools and one careless model. `execute_tool_call` validates the *arguments* and
then runs the function — in Day 1 nothing a tool did was irreversible. Watch the tasks.

In [ ]:
loop_workspace = SimulatedWorkspace()
loop_tools = workspace_tools(loop_workspace)

print("tasks before:", loop_workspace.tasks)
result = run_agent("Delete every task in my list and start fresh.", loop_tools)
print("status      :", result["status"], "| tools used:", result["tools_used"])
print("tasks after :", loop_workspace.tasks)
print("\nValidation is not authorisation. The arguments were perfectly valid; the ACTION was the")
print("problem, and nothing in Day 1's loop had an opinion about that. Section 3.7 adds the layer.")

### Try it yourself

Predict what `execute_tool_call` does with a `send_email` request that has no `body` — and what
that proves about where validation stops.

In [ ]:
# --- Worked solution ---------------------------------------------------------------
fresh = SimulatedWorkspace()
tools = workspace_tools(fresh)

bad = {"id": "t1", "name": "send_email", "arguments": {"to": "mentor@example.test", "subject": "Hi"}}
print("missing field ->", execute_tool_call(bad, tools), "| sent:", len(fresh.sent))

good = {"id": "t2", "name": "send_email",
        "arguments": {"to": "stranger@example.test", "subject": "Hi", "body": "Synthetic"}}
print("valid fields  ->", execute_tool_call(good, tools), "| sent:", len(fresh.sent))

# The malformed request never reached the function; the well-formed one to a stranger sailed
# straight through. Argument validation checks SHAPE. Whether the action is allowed at all is a
# different question, and it needs a different layer.

### Checkpoint

**1. Why treat `create_draft` and `send_email` differently when both write data?**

<details><summary>Show answer</summary>

A draft is reversible: delete it and nobody outside ever saw it. Sending leaves the machine. The question is not "does it write?" but "can it be undone?".

</details>

**2. A tool's description says it must never be used without permission. Is that a guardrail?**

<details><summary>Show answer</summary>

Not a load-bearing one. Step 2 changed state with a direct call and no description was consulted. Enforcement sits in the code that decides whether the function is called at all.

</details>

### Recap

- **Limitation seen:** valid arguments were enough to wipe the task list — through a direct call *and* through Day 1's loop.
- **Layer added:** a simulated workspace with an explicit risk class per tool, registered with schemas and argument models.
- **Evidence:** two reads changed nothing, a malformed send was refused for its shape, and a well-formed destructive call still ran.